In [1]:
import leafmap

m = leafmap.Map(center=[-4.40, 15.30], zoom=17)   # [latitude, longitude] de ta zone
m.add_basemap("Esri.WorldImagery")
m

Map(center=[-4.4, 15.3], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out…

In [30]:
m.save_draw_features("data/d.geojson")

In [31]:
import geopandas as gpd

terrains = gpd.read_file("data/d.geojson")

In [32]:
centroid = terrains.geometry.iloc[0].centroid

In [33]:
m2 = leafmap.Map(center=[centroid.y, centroid.x], zoom=17)
m2.add_basemap("SATELLITE")
m2.add_gdf(terrains ,layer_name="Parcelles", info_mode="on_hove", style={
    "fillColor": "lightgreen",
    "color" : "greens"
})
m2

Map(center=[-2.4995175, 28.873885], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

In [34]:
m2.save_draw_features("data/points.geojson")

In [35]:
import geopandas as gpd
import leafmap

terrains = gpd.read_file("data/d.geojson")
point_erosion = gpd.read_file("data/points.geojson")

crs_metrique = terrains.estimate_utm_crs()
terrains_proj = terrains.to_crs(crs_metrique)
points_proj = point_erosion.to_crs(crs_metrique)

zone_buffer_proj = points_proj.geometry.buffer(50).unary_union

terrains_proj['surface_totale_m2'] = terrains_proj.geometry.area.round(2)

geom_danger_proj = terrains_proj.geometry.intersection(zone_buffer_proj)
geom_saine_proj = terrains_proj.geometry.difference(zone_buffer_proj)

terrains_proj['surface_danger_m2'] = geom_danger_proj.area.round(2)
terrains_proj['surface_saine_m2'] = geom_saine_proj.area.round(2)
terrains_proj['pct_danger'] = ((terrains_proj['surface_danger_m2'] / terrains_proj['surface_totale_m2']) * 100).round(1)

gdf_saine = terrains_proj.copy()
gdf_saine['geometry'] = geom_saine_proj
gdf_saine = gdf_saine[~gdf_saine.is_empty].to_crs(epsg=4326)

gdf_danger = terrains_proj.copy()
gdf_danger['geometry'] = geom_danger_proj
gdf_danger = gdf_danger[~gdf_danger.is_empty].to_crs(epsg=4326)

gdf_buffer = gpd.GeoDataFrame(geometry=[zone_buffer_proj], crs=crs_metrique).to_crs(epsg=4326)
points_wgs84 = point_erosion.to_crs(epsg=4326)

m3 = leafmap.Map(center=[points_wgs84.geometry.y.mean(), points_wgs84.geometry.x.mean()], zoom=18)
m3.add_basemap("HYBRID")

m3.add_gdf(
    gdf_saine,
    layer_name="Parties Saines",
    style={"color": "#2ecc71", "fillColor": "#2ecc71", "fillOpacity": 0.5, "weight": 2},
    info_mode="on_click"
)

m3.add_gdf(
    gdf_danger,
    layer_name="Parties en Danger",
    style={"color": "#e74c3c", "fillColor": "#e74c3c", "fillOpacity": 0.7, "weight": 2},
    info_mode="on_click"
)

m3.add_gdf(
    gdf_buffer,
    layer_name="Rayon d'Érosion (14m)",
    style={"color": "#000000", "fillOpacity": 0, "weight": 2, "dashArray": "5, 5"}
)

m3.add_gdf(
    points_wgs84,
    layer_name="Point d'Érosion",
    style={"color": "#000000", "fillColor": "#ff0000", "radius": 6}
)

m3

Map(center=[np.float64(-2.499296), np.float64(28.874005)], controls=(ZoomControl(options=['position', 'zoom_in…